# Generative Sequential Recommendation — Colab 训练 (RQ-VAE 4-token)

**运行前**：Runtime → Change runtime type → **GPU**（T4 或 A100）

**需要手动上传的文件**：
- `embedding/semantic_ids_rqvae.npy`（本地已生成，~400 KB）

`data/beauty_data.pkl` 已在 GitHub 仓库中，clone 后自动存在。

本 notebook 只跑下游 GPT-2 训练 + 评估，不重新训 RQ-VAE。
如需重训 RQ-VAE，本地跑 `python embedding/rqvae.py && python embedding/generate_rqvae_ids.py`，再上传新的 `semantic_ids_rqvae.npy`。

---

## 运行顺序
1. **环境**（Cell 1–4）：GPU 检查 → 装依赖 → clone repo → 上传 semantic IDs
2. **训练 + 评估**（Cell 5–6）：`train.py` → `evaluate.py`
3. **下载结果**（Cell 7）

In [ ]:
# Cell 1：确认 GPU 可用
import torch
print(f'GPU 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    print(f'显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  未检测到 GPU，请先到 Runtime → Change runtime type → GPU')

In [ ]:
# Cell 2：安装依赖
!pip install transformers scikit-learn -q
print('依赖安装完成 ✅')

In [ ]:
# Cell 3：克隆仓库
!git clone https://github.com/rhine-yangrui/Generative-Sequential-Recommendation.git
%cd Generative-Sequential-Recommendation
!pwd && ls

In [ ]:
# Cell 4：上传 semantic_ids_rqvae.npy
# 运行后点击「Choose Files」，选择本地 embedding/semantic_ids_rqvae.npy
import os
from google.colab import files

os.makedirs('embedding', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print('请上传 semantic_ids_rqvae.npy（本地项目 embedding/ 目录里）')
uploaded = files.upload()
for fname in uploaded:
    dest = f'embedding/{fname}'
    os.rename(fname, dest)
    size_kb = os.path.getsize(dest) / 1024
    print(f'✅ 已移动到 {dest}  ({size_kb:.0f} KB)')

# 验证文件
import numpy as np
sids = np.load('embedding/semantic_ids_rqvae.npy', allow_pickle=True).item()
vals = list(sids.values())
print(f'\n✅ semantic_ids_rqvae.npy 加载成功：{len(sids)} items')
print(f'   示例：item {list(sids.keys())[0]} → {vals[0]}')
assert len(vals[0]) == 4, f'期望 4-token ID，实际 {len(vals[0])}'

ranges = list(zip(*vals))
for i, r in enumerate(ranges):
    print(f'   c{i+1} 范围: {min(r)}~{max(r)}')

unique_ids = len(set(tuple(v) for v in vals))
print(f'   唯一 4-tuple 数: {unique_ids} / {len(sids)}  {"✓ 零冲突" if unique_ids == len(sids) else "✗ 有冲突！"}')

---
## 训练生成式推荐模型（RQ-VAE 4-token + GPT-2）

In [ ]:
# Cell 5：训练
# 200 epoch + 全量 val 每 2 epoch 评估，A100 约 6-8 小时
# 常数 lr=1e-4，对齐 TIGER 参考实现（无 scheduler）
# 早停依据 validation NDCG@10
# ckpt 自动带 sid tag：
#   semantic_ids_rqvae.npy       → best_model_t5_200ep.pt
#   semantic_ids_rqvae_3kep.npy  → best_model_t5_200ep_3kep.pt
#   semantic_ids_rqvae_t5_*.npy  → best_model_t5_200ep_t5_*.pt
!python train.py

In [ ]:
# Cell 6：评估（all-rank Recall@K / NDCG@K，constrained beam search）
!python evaluate.py

---
## 下载结果

In [ ]:
# Cell 7：下载 checkpoint（自动匹配当前 ACTIVE_SEMANTIC_IDS 对应的 ckpt）
import os, glob
from google.colab import files

matches = sorted(glob.glob('checkpoints/best_model_t5_*ep*.pt'))
if not matches:
    print('⚠️  checkpoints/ 下没有 best_model_t5_*.pt')
for fpath in matches:
    files.download(fpath)
    print(f'✅ 下载：{fpath}  ({os.path.getsize(fpath)/1e6:.1f} MB)')